# ✦ NOVA — a top-to-bottom deep dive (interview edition)

> *Research, guided by SONIC.*

This notebook is **one continuous lesson**, not a file-by-file tour. We start from the
single simplest question — *"what happens when a user types a research idea and hits a
button?"* — and keep pulling that thread. Every time the story needs a new idea
(a Streamlit rerun, a LangGraph node, an `interrupt()`, a cross-encoder reranker) we stop,
learn it properly (**what it does, why it's needed, how it's used, what the alternatives were**),
then keep going.

By the end you'll be able to stand at a whiteboard and narrate NOVA end to end, defend
every design choice, and — because I'll flag them as we pass — point at the real rough
edges in the code.

**How to read this**
- 📖 Markdown cells teach. 💻 Code cells show the *real* code from your repo (lightly trimmed for flow).
- **⚠️ Bug watch** callouts mark genuine bugs / typos / dead code I noticed. I *don't* fix
  them silently — spotting them is a skill. Where I'm not 100% sure, I hedge.
- 🔌 **Runnable-standalone** cells work in any Python. 🧩 **Env-dependent** cells need NOVA's
  virtualenv (`langgraph`, `langchain-*`, `streamlit`, `sentence-transformers`, …). They're
  copy-paste-runnable *there*, in order.

---

### The 30-second mental model (read this first, everything hangs off it)

Two words carry the whole product:

- **NOVA** is the *product* — the Streamlit app.
- **SONIC** is a *persona* — a friendly cartoon researcher who narrates every step.

NOVA is really **three programs stitched into one process**, with their internal logic
*unchanged*:

1. **`app/` — the research pipeline.** Two LangGraph "agents":
   an **INTENT** graph that reshapes a messy idea into *Problem / Objective / Additional Context*,
   and a **SEARCH** graph that hits arXiv + Semantic Scholar + OpenAlex, reranks with SPECTER,
   and clusters papers by approach.
2. **`chatbot_core/` — a single-PDF Q&A bot** (`Qa.py` + `vectorizeer.py`): download a paper's
   PDF, chunk it, embed it into Chroma, answer questions grounded in the text.
3. **`nova_app.py` + `utils/` — the only new code:** the Streamlit UI and the *wiring* that
   drives the other two. It changes no agent or chatbot logic.

The user's journey (hold this the whole way):

```
raw idea
  → INTENT graph frames it            → you review/edit the framing
  → SEARCH graph fetches→rerank→cluster
  → results shown as thumbnails grouped by approach
  → "💬 Chat it out" on a paper: download its PDF → vectorize → grounded Q&A
```


## 1. The first question: what runs first, and why *that* file?

You launch with `./run.sh`, which is just:

```bash
VENV_PY=".../NOVA_venv/bin/python"
exec "$VENV_PY" -m streamlit run nova_app.py "$@"
```

So the true entry point is **`nova_app.py`**. Open it and the very first *executable* line
is not `import streamlit` — it's this, and the ordering is deliberate:


In [ ]:
# nova_app.py  (top of file)
from utils import paths  # noqa: F401  — MUST be first: wires sys.path + chdir + .env

import streamlit as st

from utils.agents import boot
from utils.state import init_session_state
from utils.styles import inject_styles
from utils.view_chat import render_chat
from utils.view_refining import render_refining
from utils.view_results import render_results
from utils.view_review import render_review
from utils.view_searching import render_searching
from utils.view_welcome import render_welcome


### Why `import utils.paths` has to be line 1

`utils/paths.py` isn't a module full of functions you *call*. It's a module whose **import
side-effects** set up the world. Importing it *runs* it, and running it does four things:

- **`os.chdir(BASE)`** — makes NOVA's folder the working directory, so every later `./vectorstores`
  and `./downloads` relative path resolves *under NOVA/*, no matter where you launched from.
- **`sys.path.insert(0, BASE)`** — so `import app...` (the research pipeline package) resolves.
- **`sys.path.insert(0, BASE/"chatbot_core")`** — so the chatbot's *flat* imports
  `import Qa` and `from vectorizeer import ...` resolve. (The chatbot was copied verbatim and
  imports itself flatly — rather than rewrite it, NOVA bends `sys.path` to it. This is the
  "logic unchanged" promise in action.)
- **`load_dotenv(BASE/".env")`** — pull API keys into `os.environ` *before* any module that
  reads them at import time gets imported.

`# noqa: F401` tells the linter "yes, this import looks unused — leave it alone." It *is*
used, just for its side-effects, not its name.


In [ ]:
# utils/paths.py  (the whole thing — it's all side-effects)
import os
import sys
from pathlib import Path

BASE = Path(__file__).resolve().parent.parent
os.chdir(BASE)                                   # ./vectorstores, ./downloads resolve under NOVA/
sys.path.insert(0, str(BASE))                    # `import app...`  (the agent package)
sys.path.insert(0, str(BASE / "chatbot_core"))   # `import Qa`, `from vectorizeer import ...`

from dotenv import load_dotenv
load_dotenv(BASE / ".env")

DOWNLOADS_DIR = BASE / "downloads"
DOWNLOADS_DIR.mkdir(exist_ok=True)
VECTORSTORES_DIR = BASE / "vectorstores"

PDF_UA = {"User-Agent": "Mozilla/5.0 (compatible; NOVA-research/1.0)"}


**"But it's imported in five other files too — doesn't it run five times?"** No, and this is
worth being able to say out loud: Python **caches modules in `sys.modules`**. The *first*
`import utils.paths` executes the file; every later `import utils.paths` anywhere just returns
the already-built module object. So the `chdir`/`sys.path`/`.env` setup happens **exactly once**,
and every other util gets the finished `BASE`, `DOWNLOADS_DIR`, etc. for free. That module cache
is the same mechanism that makes the singletons later in this notebook (loaded LLMs, the SPECTER
model) load once.

**The alternative** would be a `sys.path` hack or `PYTHONPATH` env var in `run.sh`, or making
both sub-projects installable packages (`pip install -e`). The chosen approach keeps the
sub-projects byte-for-byte unmodified and self-contained in one repo — the right call for a
"stitch two things together without touching them" job.

> **⚠️ Bug watch (doc drift, harmless):** `nova_app.py`'s module docstring and the README both
> describe the research pipeline as living in `agent/`, but the actual folder is `app/`. The code
> imports `app...` correctly; only the prose is stale. Small, but exactly the kind of thing an
> interviewer might poke at — "your README says `agent/`, where is it?"


## 2. The one Streamlit fact that explains the entire architecture

Everything odd about NOVA's structure — the caching, the "run-once guards," the stage machine —
falls out of a **single** Streamlit rule:

> **Every user interaction re-runs the whole script from top to bottom.**

Click a button, type in a box, anything → Streamlit executes `nova_app.py` again, start to
finish, like it's the first time. There's no callback system, no persistent widget tree you
mutate. The "app" is a *pure function from state → UI* that Streamlit calls over and over.

Two consequences drive the rest of the codebase:

1. **Anything expensive must be cached**, or you'd reload a 440 MB embedding model on every
   keystroke. → `@st.cache_resource` (Section 4).
2. **Anything that should happen once per query needs a guard**, because the script body for a
   stage can execute several times before it transitions. → the `_intent_done_for` / `_search_done_for`
   flags (Section 6).

Here's the bottom of `nova_app.py` — the **router**. Read it as: *"given the current stage in
session_state, which single view do I render this run?"*


In [ ]:
# nova_app.py  (bottom — the router / finite-state machine)
st.set_page_config(page_title="NOVA · Research Assistant", page_icon="✦", layout="wide")
inject_styles()          # inject the CSS theme once

init_session_state()     # ensure every key we rely on exists (Section 3)

st.markdown("<div class='nova-brand'>…NOVA header HTML…</div>", unsafe_allow_html=True)

boot()                   # one-time heavy model loads behind a splash (Section 4)

# --- ROUTER ---
if st.session_state.error:
    st.warning(f"Something went sideways: {st.session_state.error}")
    st.session_state.error = ""

stage = st.session_state.stage

if stage == "results" and st.session_state.active_chat:
    render_chat()
elif stage == "welcome":
    render_welcome()
elif stage == "refining":
    render_refining()
elif stage == "review":
    render_review()
elif stage == "searching":
    render_searching()
elif stage == "results":
    render_results()
else:
    render_welcome()


### The stage machine

`stage` is a plain string in `st.session_state` that walks a fixed path:

```
welcome ──► refining ──► review ──► searching ──► results ──► (chat overlays results)
   ▲                                                  │
   └──────────────────── "Start over" ────────────────┘
```

Each view function does its work and, when it's time to advance, sets
`st.session_state.stage = "<next>"` and calls `st.rerun()` to *immediately* re-execute the
script — which lands in the next branch of this router. So the whole UI is a **finite-state
machine** where the transition is "mutate `stage`, then rerun."

Notice the **order** of the checks: the chat branch is tested first (`stage == "results" and
active_chat`) so that opening a chat *overlays* the results view without changing `stage` away
from `"results"` — hit "← Back to papers" and you're exactly where you left off. The trailing
`else: render_welcome()` is a defensive catch-all for an unknown stage value.

`unsafe_allow_html=True` appears everywhere: Streamlit escapes HTML in `st.markdown` by default
(XSS safety); NOVA opts out because it renders its own styled cards, badges, and the SONIC SVG.
Fine here — the HTML is all app-authored, never user input.


## 3. Session state: the memory that survives reruns

If the script reruns top-to-bottom every time, where does *anything* persist? In
**`st.session_state`** — a dict-like object scoped to one browser session that Streamlit keeps
across reruns. It's the app's entire memory.

`utils/state.py` declares the schema up front (great for readability — you see every field the
app can hold in one place) and initializes it:


In [ ]:
# utils/state.py
import shutil
import streamlit as st
from utils.paths import DOWNLOADS_DIR, VECTORSTORES_DIR

DEFAULTS = {
    "stage": "welcome",          # welcome | refining | review | searching | results
    "run_id": "",
    "user_query": "",
    "problem": "",
    "objective": "",
    "context": "",
    "clusters": [],
    "papers_by_key": {},         # normalized_title -> full record (flattened, for chat lookup)
    "active_chat": None,         # normalized_title of the paper being chatted, or None
    "chats": {},                 # normalized_title -> {retriever, chain, messages, title}
    "source_status": {},         # {"Semantic Scholar": {"state": "rate_limited", ...}, ...}
    "error": "",
    "_intent_done_for": "",      # run-once guards (keyed by run_id)
    "_search_done_for": "",
}

def init_session_state():
    for k, v in DEFAULTS.items():
        st.session_state.setdefault(k, v)   # only sets if the key is absent


### Why `setdefault` and not `=`

`st.session_state.setdefault(k, v)` writes the default **only if the key doesn't already exist.**
On the *first* run it seeds everything; on *every later* run it's a no-op, so it never clobbers
the user's in-progress query or results. If it used plain assignment (`st.session_state[k] = v`),
every rerun would wipe the app back to the welcome screen — the classic Streamlit beginner bug.

Now the reset path. Two functions, deliberately separated:


In [ ]:
# utils/state.py  (continued)
def wipe_disk_cache():
    """Delete every cached vectorstore + downloaded PDF so a new search starts clean —
    no old paper's chunks or PDFs can leak in."""
    for folder in (VECTORSTORES_DIR, DOWNLOADS_DIR):
        try:
            if folder.exists():
                shutil.rmtree(folder, ignore_errors=True)
            folder.mkdir(exist_ok=True)
        except Exception:
            pass

def reset_all(wipe_cache: bool = True):
    """Full memory refresh: clear this session's results + open chats, and
    (by default) wipe the on-disk vectorstore/PDF caches too."""
    for k, v in DEFAULTS.items():
        st.session_state[k] = v if not isinstance(v, (list, dict)) else type(v)()
    st.session_state["stage"] = "welcome"
    if wipe_cache:
        wipe_disk_cache()


**One subtle, important line:**

```python
st.session_state[k] = v if not isinstance(v, (list, dict)) else type(v)()
```

For scalar defaults (`""`, `None`) it reuses the default object `v`. For **mutable** defaults
(`[]`, `{}`) it builds a *fresh* empty container with `type(v)()` (`list()` or `dict()`). Why the
fuss? Because if it wrote `st.session_state[k] = v` for a list/dict, every reset would point
session_state at *the very same* `DEFAULTS["clusters"]` list object. Mutating results later would
then silently mutate the shared default — the infamous **mutable-default-argument** footgun, one
level up. Copying via `type(v)()` guarantees isolation. Clean, defensive code — call this out in
an interview, it shows you understand aliasing.

Notice `reset_all` walks `DEFAULTS`, so keys **not** in `DEFAULTS` — like `_booted` (Section 4) —
survive a reset. That's intentional: "Start over" should forget your *results*, not re-download
half a gigabyte of models.


## 4. Loading heavy things once: `@st.cache_resource`

Recall consequence #1 of the rerun model: **never reload expensive things.** NOVA has three:
the compiled LangGraph agents, the SPECTER reranker (loaded as a side-effect of importing the
search graph), and the chatbot's embedding + cross-encoder models. `utils/agents.py` guards all
of them.


In [ ]:
# utils/agents.py
import streamlit as st

@st.cache_resource(show_spinner=False)
def load_agents():
    """Import the compiled LangGraph agents. Importing the search graph also
    loads the SPECTER reranker model at module import time (by design)."""
    from app.modules.intent.graph import graph as intent_graph
    from app.modules.search.graph import graph as search_graph
    return intent_graph, search_graph

@st.cache_resource(show_spinner=False)
def warm_chatbot_models():
    """Pre-load the chatbot's embedding + cross-encoder models at startup so the
    first 'Chat it out' click doesn't pay the model-load cost."""
    from vectorizeer import get_embeddings
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    embeddings = get_embeddings()
    reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
    return embeddings, reranker

def boot():
    """Run the one-time heavy loads behind a single friendly splash."""
    if st.session_state.get("_booted"):
        return
    with st.spinner("✦ Waking up SONIC — loading the research + reading models…"):
        load_agents()
        warm_chatbot_models()
    st.session_state["_booted"] = True


### `@st.cache_resource` vs `@st.cache_data` — know the difference cold

Streamlit has two caching decorators and interviewers love the distinction:

- **`@st.cache_resource`** — for **unserializable, shared, singleton objects**: DB connections,
  ML models, compiled graphs. The cache stores the *actual object* and hands the **same instance**
  to every session and every rerun. No copying. That's exactly what you want for a 440 MB model
  or a LangGraph with a live in-memory checkpointer.
- **`@st.cache_data`** — for **serializable return values** (DataFrames, dicts, API JSON). It
  *copies* the result on each hit, so callers can't mutate the cache. Wrong choice for a model
  (you can't/shouldn't copy a torch model).

`show_spinner=False` suppresses each function's own little spinner because `boot()` wraps both in
**one** friendly "Waking up SONIC…" splash — better UX than two stacked spinners.

### The two-layer "load once" trick

There are actually **two** guards stacked here, and understanding both is the point:

1. `boot()` uses a **session flag** `_booted` so the *UI-level* splash logic runs once per session.
2. `load_agents()` / `warm_chatbot_models()` use `@st.cache_resource` so the *actual heavy work*
   runs once per **server process** — shared across *all* sessions, and re-callable cheaply.

That second layer is why `render_refining` and `render_searching` can each call `load_agents()`
again with zero cost — they get the cached graphs instantly. `boot()` is just the *warm-up
trigger* so the cost is paid up front behind a splash, not lazily on the user's first real action.

### The sneakiest line in the whole app

```python
from app.modules.search.graph import graph as search_graph
```

Importing the search graph **loads the SPECTER embedding model**, because `reranking.py`
(imported at the top of `graph.py`) does `_MODEL = SentenceTransformer(...)` **at module scope**,
not inside a function. So a plain `import` triggers a multi-second model download/load. That's a
deliberate pattern — pay the cost once at boot — but it's *invisible* unless you know import-time
side-effects are in play. We'll see the same trick used on purpose in `jobs.py` later.

> **⚠️ Bug watch (dead-ish, low severity):** `warm_chatbot_models()` builds `embeddings` and
> `reranker` and returns them — but nothing ever consumes that return value, and the real chat
> path (`get_embeddings()` in `vectorizeer`, a fresh `HuggingFaceCrossEncoder` in `Qa.get_retriever`)
> **re-instantiates** its own copies. So the warm-up helps *only* by populating the HuggingFace/
> torch on-disk + process caches (which the fresh instances then hit instantly). That's the stated
> intent, so it's not *wrong* — but the returned tuple is genuinely unused, and a reader could
> reasonably expect those exact objects to be reused. Worth naming.


## 5. The INTENT graph — learning LangGraph from the ground up

Now the first real "agent." Forget frameworks for a second and state the goal in plain terms:

> *Take a rambling research idea and turn it into a clean, three-part brief — Problem, Objective,
> Additional Context — that a search pipeline can act on. Let a human approve/edit it before we
> spend money searching.*

NOVA does this with **LangGraph**. Before the code, the three ideas you need:

1. **State** — a single typed dict that flows through the whole graph. Each step reads from it and
   returns a *partial* update that gets merged in.
2. **Nodes** — plain Python functions `state -> partial_state`. A node is just "do a thing, return
   what changed."
3. **Edges** — wiring that says which node runs after which. Fan-**out** (one → many) runs branches
   in parallel; fan-**in** (many → one) waits for all of them.

Why a graph at all, instead of one big function that calls the LLM four times? Because a graph gives
you, for free: **parallelism** (frame Problem/Objective/Context simultaneously), **a persistent
checkpoint** you can pause and resume (the human-in-the-loop review), and **streaming** of
per-node progress (the live checklist later). Those three are exactly what a good research UX needs.

Here's the state and the shared LLM/tool setup:


In [ ]:
# app/modules/intent/graph.py  (state + shared clients)
from typing import TypedDict
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from app.core.logging_config import get_node_logger

load_dotenv()

class ResearchIntentState(TypedDict):
    run_id: str
    user_query: str
    problem: str
    objective: str
    additional_context: str
    aggregated_intent: str
    search_results: str
    polished_research_intent: str
    human_verified_intent: str

second_api_key = os.getenv("SECOND_GROQ_API_KEY")

# LLM + tools — instantiated once at import time, reused across requests
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=second_api_key)
search_tool = TavilySearch(max_results=5)


### Why `TypedDict` for the state?

`ResearchIntentState` is a **`TypedDict`**: a dict at runtime, but with a declared shape so type
checkers and readers know exactly which keys exist and their types. LangGraph uses that declaration
to know the state's fields and how to merge node outputs into it. A node returns `{"problem": ...}`
and LangGraph merges just that key into the running state.

**Alternatives you should be able to name:** LangGraph also accepts a **Pydantic `BaseModel`**
(adds runtime validation + defaults, at some cost) or a **dataclass**. `TypedDict` is the
lightest — zero runtime overhead, no validation — which is why it's the common default. NOVA uses
`TypedDict` for graph *state* but Pydantic for *LLM structured output* (Section 8) and *API
request/response* (Section 9) — a nice illustration of picking the right tool per layer.

The `llm` and `search_tool` are built **once at import time** (module scope), not per call — the
same "construct expensive clients once" instinct as the cached models. Because the module is
imported once (module cache again), these are effectively process-wide singletons.


### A node is just a function. Here are two of them.

The intent graph frames the idea in three independent nodes. Each is the same shape: build a
tightly-scoped prompt, call the LLM, log, return the one field it owns. Look at how *narrow* each
prompt's job is — that's the whole design philosophy.


In [ ]:
# app/modules/intent/graph.py  (two of the three framing nodes, prompts trimmed)
def problem_framing(state: ResearchIntentState) -> ResearchIntentState:
    logger = get_node_logger(state["run_id"], "intent.problem")
    prompt = f"""You are extracting the PROBLEM STATEMENT from a researcher's raw idea.
Your only job: describe what is happening — the gap/limitation/open issue — as understood
from the query.
STRICT RULES:
- Do NOT propose what should be done about it.  Do NOT suggest methods or solutions.
- Do NOT include constraints, preferences, or scope limits.
- Write 2-4 sentences, formal academic register, third person.
Output only the problem statement text. No labels, no preamble.

User's raw query:
{state['user_query']}
"""
    logger.info("Framing problem statement (%d chars)", len(state["user_query"]))
    try:
        problem = llm.invoke(prompt).content
    except Exception:
        logger.exception("LLM call failed while framing the problem statement")
        raise
    return {"problem": problem}          # <-- returns ONLY the field it owns

def objective_framing(state: ResearchIntentState) -> ResearchIntentState:
    logger = get_node_logger(state["run_id"], "intent.objective")
    prompt = f"""You are extracting the RESEARCH OBJECTIVE...
- Phrase as an OUTCOME ("identify", "evaluate", "compare"), never a specific technique.
  Wrong: "Use contrastive learning to reduce hallucinations."
  Right: "Identify techniques that reduce hallucinations while preserving answer quality."
...
User's raw query:
{state['user_query']}
"""
    objective = llm.invoke(prompt).content
    return {"objective": objective}


### Why split one idea into three separate LLM calls?

You could ask one prompt "give me problem, objective, and context." NOVA deliberately doesn't.
Three **single-responsibility** prompts each get to be long, strict, and un-distracted — the
Problem prompt is *obsessed* with "don't leak solutions," the Objective prompt is *obsessed* with
"outcomes, not techniques." A combined prompt has to juggle all those rules at once and reliably
does each one worse. This is **prompt decomposition**, and it maps beautifully onto graph nodes:
one concern per node.

The bonus: because the three share no data (each reads only `user_query`), they can run
**in parallel**. That's the fan-out we wire up next.

Also note the **error pattern**: `try/except` → `logger.exception(...)` → `raise`. It logs the
full traceback to that run's log file, then re-raises so the failure isn't swallowed — the graph
aborts and the UI's outer `try/except` (in `render_refining`) shows a friendly error. Log-and-
re-raise, not log-and-continue: the difference between "observable failure" and "silent corruption."


### The `aggregator` and `web_polish` nodes — sequential refinement

After the three framings, `aggregator` harmonizes them into one document (consistent terminology,
smoothed transitions) — crucially **without moving content between sections**. Then `web_polish_node`
is a richer, *multi-step* node: it (1) asks the LLM for a short web query, (2) runs it through the
**Tavily** search tool, (3) feeds the results back to the LLM to correct terminology and entity
names — while forbidding it from changing meaning or adding citations.


In [ ]:
# app/modules/intent/graph.py  (web_polish_node — a 3-step node)
def web_polish_node(state: ResearchIntentState) -> ResearchIntentState:
    logger = get_node_logger(state["run_id"], "intent.polish")

    # Step 1: derive a concise search query from the aggregated intent
    query_gen_prompt = f"""Given this Research Intent, generate ONE concise web search query
(under 15 words) ... Output only the query text.

Research Intent:
{state['aggregated_intent']}
"""
    search_query = llm.invoke(query_gen_prompt).content

    # Step 2: run the search tool
    raw_results = search_tool.invoke({"query": search_query})
    results = raw_results.get("results", [])
    results_text = "\n\n".join(
        f"Source: {r.get('title', 'N/A')}\n{r.get('content', '')}" for r in results
    )

    # Step 3: polish the intent using the results, with strict "don't change meaning" boundaries
    polish_prompt = f"""You are GROUNDING and POLISHING a Research Intent using web results.
... Preserve the exact three-section format with headers: Problem / Objective / Additional Context.

Current Research Intent:
{state['aggregated_intent']}

Web Search Results:
{results_text}
"""
    polished = llm.invoke(polish_prompt)
    return {"search_results": results_text, "polished_research_intent": polished.content}


A node isn't limited to one LLM call — it's arbitrary Python. `web_polish_node` chains
*LLM → tool → LLM*. This is the "**augmented LLM**" idea (an LLM given tools/retrieval) living
inside a single graph node. Tavily is an LLM-oriented search API; `search_tool.invoke({"query": ...})`
returns structured results NOVA reshapes into a plain text block for the final prompt.

Notice the recurring contract: **"Preserve the exact three-section format: Problem / Objective /
Additional Context."** That literal shape is a *protocol* between this graph and both the UI and
the downstream search graph. Hold that thought — Section 7 is entirely about honoring it.


### Human-in-the-loop: `interrupt()`, checkpointers, and `Command(resume=...)`

Here's the feature that justifies using LangGraph instead of a `for` loop. After polishing, we
must **pause**, show the human the draft, let them edit, and only then continue. LangGraph does
this with **`interrupt()`**.


In [ ]:
# app/modules/intent/graph.py  (the pause node + graph assembly)
def human_review_node(state: ResearchIntentState) -> ResearchIntentState:
    logger = get_node_logger(state["run_id"], "intent.human_review")
    logger.info("Pausing for human review of the polished research intent")
    human_response = interrupt({                       # <-- graph SUSPENDS here
        "polished_research_intent": state["polished_research_intent"],
        "instruction": "Review and edit this Research Intent.",
    })
    logger.info("Human-verified intent received (%d chars)", len(human_response))
    return {"human_verified_intent": human_response}   # <-- runs only AFTER resume

# --- assemble the graph ---
builder = StateGraph(ResearchIntentState)
builder.add_node("problem", problem_framing)
builder.add_node("objective", objective_framing)
builder.add_node("context", additional_context_framing)
builder.add_node("aggregator", aggregator)
builder.add_node("polish", web_polish_node)
builder.add_node("human_review", human_review_node)

builder.add_edge(START, "problem")        # fan-OUT: all three framings start together
builder.add_edge(START, "objective")
builder.add_edge(START, "context")
builder.add_edge("problem", "aggregator") # fan-IN: aggregator waits for all three
builder.add_edge("objective", "aggregator")
builder.add_edge("context", "aggregator")
builder.add_edge("aggregator", "polish")
builder.add_edge("polish", "human_review")
builder.add_edge("human_review", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)


**How `interrupt()` actually works** — this trips people up, so let's be precise:

- When execution reaches `interrupt(payload)`, LangGraph **saves the entire graph state to the
  checkpointer** and *stops*. The `.invoke()` call returns, and the returned dict contains a
  special key **`__interrupt__`** holding your payload. `human_response` is **not yet assigned** —
  the function is frozen mid-line.
- Later, you call `graph.invoke(Command(resume=value), config=config)` with the **same
  `thread_id`**. LangGraph reloads the checkpoint, and `interrupt()` **returns `value`** — so
  `human_response = value` and the node finishes from where it stopped.

Three things make this work, and you should be able to point at each:

1. **The checkpointer** (`MemorySaver`). It persists the paused state between the two `.invoke`
   calls. Without it, `interrupt()` has nowhere to save and there's nothing to resume.
2. **`thread_id`** in `config={"configurable": {"thread_id": run_id}}`. It's the key that ties the
   *resume* call back to the *right* paused conversation. NOVA uses the query's `run_id` as the
   `thread_id`.
3. **`Command(resume=...)`**. The signal "here's the value `interrupt()` should return; continue."

**`MemorySaver` — and its honest limitation.** It keeps checkpoints in **process memory only.**
Perfect for one Streamlit process serving one researcher (NOVA's exact scope). But state is lost
on restart and *not* shared across multiple workers. The code's own comment says to swap in
`SqliteSaver` / `PostgresSaver` for durability — good sign of an author who knows the trade-off
rather than one who didn't notice it.

### The fan-out / fan-in you just wired

```
        ┌─► problem ──┐
START ──┼─► objective ─┼─► aggregator ─► polish ─► human_review ─► END
        └─► context ──┘         (waits for all 3)      (pauses)
```

`START` has three outgoing edges → the three framing nodes run **concurrently**. `aggregator` has
three incoming edges → LangGraph **automatically barrier-syncs**: it won't run `aggregator` until
all three predecessors finish. You didn't write any threading — the *edge topology* declares the
parallelism, and the runtime schedules it. This is the big reason to prefer a graph over
imperative code: concurrency and synchronization become **declarative.**


### How NOVA *drives* this graph across two UI stages

The graph pauses in the middle; NOVA's stage machine straddles that pause. **`render_refining`**
runs the graph *up to* the interrupt and parks the draft into the review form:


In [ ]:
# utils/view_refining.py
import streamlit as st
from utils.agents import load_agents
from utils.intent_text import split_intent_sections
from utils.sonic import sonic_says

def render_refining():
    sonic_says("that seems great — lemme juss refine it ✨")
    run_id = st.session_state.run_id
    # Guard: run the intent graph exactly once per query, even if this stage's
    # script body re-executes before it transitions.
    if st.session_state.get("_intent_done_for") == run_id:
        return
    intent_graph, _ = load_agents()
    config = {"configurable": {"thread_id": run_id}}
    try:
        with st.spinner("Framing the problem, objective & context…"):
            result = intent_graph.invoke(
                {"user_query": st.session_state.user_query, "run_id": run_id},
                config=config,
            )
        interrupt_payload = result["__interrupt__"][0].value
        problem, objective, context = split_intent_sections(
            interrupt_payload["polished_research_intent"]
        )
        st.session_state.problem = problem
        st.session_state.objective = objective
        st.session_state.context = context
        st.session_state["_intent_done_for"] = run_id
        st.session_state.stage = "review"
    except Exception as e:
        st.session_state.error = f"{type(e).__name__}: {e}"
        st.session_state.stage = "welcome"
    st.rerun()


There's the **run-once guard** in the flesh (consequence #2 of the rerun model). Reaching the
`refining` stage could execute this body more than once before `stage` flips; `_intent_done_for ==
run_id` makes the expensive `graph.invoke` fire exactly once per query. It's keyed by `run_id` (a
fresh UUID per query) rather than a bare boolean, so a *new* query — new `run_id` — naturally
re-arms the guard. Elegant: the guard resets itself just by the key changing.

`result["__interrupt__"][0].value` is the payload we passed to `interrupt(...)`. It's a *list*
because a graph can hit multiple interrupts; NOVA's has one, so `[0]`. We then split the polished
text into three editable fields (Section 7) and advance to `review`.

Later, after the human edits and clicks "Find the papers," **`render_searching`** *resumes* the
same paused graph — same `thread_id` — with the edited text, then launches the search graph:


In [ ]:
# utils/view_searching.py  (the resume half; search-streaming shown in Section 8)
from langgraph.types import Command
from utils.intent_text import join_intent_sections

def render_searching():
    run_id = st.session_state.run_id
    if st.session_state.get("_search_done_for") == run_id:   # its own run-once guard
        return
    intent_graph, search_graph = load_agents()
    config = {"configurable": {"thread_id": run_id}}
    edited_intent = join_intent_sections(
        st.session_state.problem, st.session_state.objective, st.session_state.context
    )
    with st.spinner("Finalizing your research intent…"):
        resume_result = intent_graph.invoke(Command(resume=edited_intent), config=config)
    human_verified_intent = resume_result["human_verified_intent"]
    # ... then stream the SEARCH graph (Section 8) ...


Follow the `thread_id` and it clicks into place: `render_refining` invokes the graph and it
parks at the interrupt (checkpoint saved under `run_id`). The user edits in `render_review`.
`render_searching` calls `Command(resume=edited_intent)` with the **same** `run_id`, LangGraph
reloads that checkpoint, `human_review_node`'s `interrupt()` returns the edited text, the node
finishes, and `human_verified_intent` comes back. **One paused agent, resumed across three screens
and multiple full script reruns** — held together entirely by `thread_id` + the cached, in-memory
checkpointer. This is the single most impressive thing to be able to narrate about NOVA.


## 6. The text ⇄ form contract (a tiny module doing important work)

The graph speaks in *one string* with three headed sections. The UI wants *three editable boxes*.
And the downstream search graph's prompts assume the **exact** `Problem: / Objective: / Additional
Context:` shape. `utils/intent_text.py` is the translator that guarantees the round-trip. This one
is fully **🔌 runnable-standalone** — run it and watch:


In [ ]:
# utils/intent_text.py  (verbatim) — RUNNABLE STANDALONE
import re

_SECTION_PATTERN = re.compile(
    r"Problem:\s*(.*?)\n\s*Objective:\s*(.*?)\n\s*Additional Context:\s*(.*)",
    re.S,
)

def split_intent_sections(text: str):
    m = _SECTION_PATTERN.search(text or "")
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    return (text or "").strip(), "", ""

def join_intent_sections(problem: str, objective: str, additional_context: str) -> str:
    ac = additional_context.strip() or "None specified."
    return (
        f"Problem:\n{problem.strip()}\n\n"
        f"Objective:\n{objective.strip()}\n\n"
        f"Additional Context:\n{ac}"
    )

# --- demo the round-trip ---
draft = """Problem:
Human vs RL car-following fuel efficiency is hard to compare because many variables move at once.

Objective:
Identify a robust methodology to compare fuel efficiency across the two driving modes.

Additional Context:
None specified."""

p, o, c = split_intent_sections(draft)
print("PROBLEM  :", p)
print("OBJECTIVE:", o)
print("CONTEXT  :", c)
print("\n--- rejoined ---\n")
print(join_intent_sections(p, o, c))


### Reading the regex like an engineer

- `re.S` (DOTALL) makes `.` match newlines, so a multi-line Problem body is captured whole.
- `(.*?)` is **lazy** — it grabs the *smallest* text up to the next header, so the Problem capture
  stops at `Objective:` instead of greedily swallowing the rest.
- `\n\s*Objective:` tolerates whatever blank space sits between sections (the `join` puts a blank
  line there; an LLM might vary it).
- **The graceful fallback:** if the text doesn't match (LLM forgot a header, reordered sections),
  `split_intent_sections` returns `(whole_text, "", "")` instead of throwing — the user sees their
  text in the Problem box rather than a crash. Defensive parsing of LLM output is a recurring NOVA
  theme; you'll see it again in clustering (Section 8).
- `join_intent_sections` re-imposes the canonical shape and substitutes `"None specified."` for an
  empty context — matching what the graph's prompts expect verbatim. The two functions are
  deliberate inverses, and *both* graphs' prompts are written against this exact format string.

**Why regex over `str.split("Objective:")`?** Splitting on literal labels is brittle if a label
ever appears inside the body text, and it can't express "optional whitespace" cleanly. A single
compiled pattern states the whole structural contract in one place and either matches it or falls
back — easier to reason about and to change.


## 7. The SEARCH graph — three sources, one uniform record

The human-approved intent now feeds the second agent. Same LangGraph vocabulary, bigger machine:

```
        ┌─► arxiv ───────────┐
START ──┼─► semantic_scholar ─┼─► aggregation ─► reranker ─► clustering ─► END
        └─► open_alex ───────┘   (dedup/merge)   (SPECTER)   (LLM groups)
```

Three provider nodes fan out (parallel API calls), fan in to an aggregator that dedups across
sources, a SPECTER reranker keeps the 15 most relevant, and an LLM clusters them by approach.
Here's the state — and one design decision worth its own paragraph:


In [ ]:
# app/modules/search/graph.py  (state)
class ResearchSearchState(TypedDict):
    run_id : str
    ResearchIntent : str
    Open_Alex_paper: List[dict]
    Semantic_Scholar_paper : List[dict]
    arXiv_paper: List[dict]
    aggregator : dict
    reranked_papers : dict
    clustered_papers : List[dict]
    # Per-source status so the UI can surface WHY a source came back empty.
    # Distinct keys per source -> no concurrent-write conflict across the
    # parallel fetch nodes.
    arxiv_status : dict
    semantic_scholar_status : dict
    open_alex_status : dict


### Why each source writes to its *own* state key

The three provider nodes run **concurrently** and each returns a partial state update. LangGraph
merges those updates into the shared state. If all three tried to write the **same** key (say a
single `status` dict), you'd have two branches writing one field in the same super-step — a
**concurrent-write conflict**. LangGraph would demand you define a **reducer** (a function that
says how to combine two writes to that key, like `operator.add` for lists).

NOVA sidesteps that entirely by giving each source a **distinct key** (`arxiv_status`,
`semantic_scholar_status`, `open_alex_status`, and `arXiv_paper` / `Semantic_Scholar_paper` /
`Open_Alex_paper`). Disjoint keys → no conflict → no reducer needed. It's the simplest correct
answer to parallel writes. (Contrast: a chat-message graph that *does* want to append from multiple
nodes would use `Annotated[list, operator.add]` as a reducer — good to mention you know both paths.)

> **⚠️ Bug watch (dead code / redundancy):** Two lines at the top of this file:
> ```python
> second_api_key = os.getenv("SECOND_GROQ_API_KEY")
> api_key = os.getenv("GROQ_API_KEY")          # <-- assigned, never used
> llm   = ChatGroq(model='openai/gpt-oss-120b', api_key=second_api_key)
> llm_2 = ChatGroq(model='openai/gpt-oss-120b', api_key=second_api_key)   # <-- identical to llm
> ```
> `api_key` is read and never referenced again — dead variable. And `llm_2` is a **byte-for-byte
> duplicate** of `llm` (same model, same key); `clustering_node` uses `llm_2` while the query
> nodes use `llm`, but since they're identical, `llm_2` could just be `llm`. Neither breaks
> anything; both are the kind of "leftover from earlier iteration" cruft a sharp reviewer points at.


### One prompt per API, because each API matches text differently

The three provider nodes look similar but their prompts are **radically** different, and the
*reason* is the single most interview-worthy insight in the search graph:

- **arXiv** is *purely lexical* with field prefixes (`ti:`, `abs:`, `all:`) and `AND/OR/ANDNOT`.
  So its prompt says: extract 4–6 concrete noun phrases, keep the boolean structure flat, don't
  nest prefixes.
- **Semantic Scholar** implicitly **ANDs** every term against title+abstract — one wrong required
  word zeroes the whole query. So its prompt says: require **at most 2** core concepts, and hedge
  uncertain terminology as `("human-driven" | "human-operated" | "manually driven")` OR-groups.
- **OpenAlex** also defaults to AND across titles/abstracts/fulltext. Its prompt says: 2–4 concept
  groups, quote every multi-word phrase, OR-group synonyms inside each concept.

Same research intent, three *dialects* of query, because the retrieval semantics differ. If you
handed all three the same keyword string, you'd get wildly uneven recall. The LLM is used here as
a **query translator per backend** — a genuinely clever use of it. Here's the arXiv node; the
other two are the same skeleton with a different prompt:


In [ ]:
# app/modules/search/graph.py  (arXiv provider node)
def arXiv_papers(state: ResearchSearchState) -> ResearchSearchState:
    logger = get_node_logger(state["run_id"], "search.arxiv")
    prompt = f"""You are converting a Research Intent into a search query for the arXiv API.
arXiv search is PURELY LEXICAL — field-prefixed keyword overlap only.
- Use only ti: / abs: / all: prefixes; default to all:.
- Combine with UPPERCASE AND, OR, ANDNOT only. Keep structure FLAT (no nested prefixes).
- Extract only concrete technical noun phrases from Problem + Objective (ignore Additional Context —
  arXiv can't reason about constraints).
- Join multi-word phrases with + (arXiv convention). Produce 4-6 core terms max.
- Output ONLY the final query string.

Research Intent:
{state['ResearchIntent']}"""
    query = llm.invoke(prompt).content
    query = query.strip().strip('"').strip("'")           # strip stray quoting from the LLM
    logger.info("arXiv query: %s", query)
    papers = search_arxiv(query=query, logger=logger)
    return {'arXiv_paper': papers, 'arxiv_status': {"state": "ok", "count": len(papers)}}


The `.strip().strip('"').strip("'")` is a small but telling defensive move: LLMs love to wrap
their "output only the query" answer in quotes anyway, so we peel them off. Every provider node does
this. (You'll notice the arxiv node hard-codes `arxiv_status = {"state":"ok",...}` — it can't
distinguish a rate-limit, unlike the other two, because `search_arxiv` returns `[]` on any HTTP
failure without a `status_out` channel. Minor asymmetry, not a bug.)

> **⚠️ Bug watch (fragile, probably harmless):** The prompt tells the LLM to join phrases with `+`
> (`reinforcement+learning`), and then `search_arxiv` *also* runs `query.replace(" ", "+")`. So
> spaces around the booleans become `+AND+`, giving strings like `all:reinforcement+learning+AND+all:fuel`.
> arXiv treats `+` as a space in the query, so this *does* still parse — but the double `+`
> convention (LLM-inserted **and** code-inserted) is the kind of thing that silently breaks if
> either side changes. I'd flag it as fragile rather than outright broken; hedge here since it works
> in practice.


### Inside a provider: arXiv returns XML, so we parse Atom by hand

Each provider's job: hit the API, validate every record through the shared `ResearchPaper` Pydantic
model, return a list of clean dicts. arXiv is the simplest — it returns an **Atom XML** feed, parsed
with the stdlib `xml.etree.ElementTree`:


In [ ]:
# app/modules/search/providers/arxiv.py  (core of it, trimmed)
import xml.etree.ElementTree as ET
import requests
from app.modules.search.providers.models import ResearchPaper

def search_arxiv(query, max_results=10, logger=None):
    query = query.replace(" ", "+")
    url = (f"https://export.arxiv.org/api/query?search_query={query}"
           f"&start=0&max_results={max_results}&sortBy=relevance&sortOrder=descending")
    try:
        response = requests.get(url, timeout=15)
    except requests.RequestException as e:
        logger.warning("arXiv request failed: %s", e); return []      # network fail -> [] not crash
    if response.status_code != 200:
        return []
    try:
        root = ET.fromstring(response.content)
    except ET.ParseError as e:
        return []                                                      # bad XML -> [] not crash

    namespace = {"atom": "http://www.w3.org/2005/Atom"}
    papers = []
    for entry in root.findall("atom:entry", namespace):
        title_el = entry.find("atom:title", namespace)
        if title_el is None or not (title_el.text or "").strip():
            continue                                                   # no title -> skip this entry
        # ... pull summary(abstract), authors, published->year, the link with title="pdf", id->url ...
        try:
            paper = ResearchPaper(title=title, authors=authors or None, abstract=abstract,
                                  year=year, citation_count=None, url=url_val,
                                  pdf_url=pdf_link, source="arXiv")
            papers.append(paper.model_dump())
        except Exception as e:
            logger.warning("Skipped malformed arXiv entry: %s", e)
            continue                                                   # one bad entry != whole batch dies
    return papers


### The `ResearchPaper` model — the *lingua franca* of the pipeline

Three different APIs return three totally different shapes. The pipeline can't reason about that
chaos, so every provider funnels its records through **one Pydantic model** that defines the
canonical paper shape:


In [ ]:
# app/modules/search/providers/models.py  (verbatim)
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class ResearchPaper(BaseModel):
    title: str = Field(description="The title of the paper")
    authors: Optional[List[str]] = Field(default=None)
    abstract: Optional[str] = Field(default=None)
    year: Optional[int] = Field(default=None)
    citation_count: Optional[int] = Field(default=None)
    url: Optional[str] = Field(default=None)
    pdf_url: Optional[str] = Field(default=None)
    source: Literal["arXiv", "SemanticScholar", "OpenAlex"] = Field(description="source name")


**Why Pydantic here (but `TypedDict` for graph state)?** Because provider data is *external and
untrusted* — the API might send a string where you expect an int, or omit a field. Pydantic
**validates and coerces at construction time**: `ResearchPaper(...)` raises if the data doesn't fit,
which is why each provider wraps it in `try/except` and skips the one bad record instead of crashing
the batch. `Literal[...]` on `source` even constrains it to exactly three allowed strings. Graph
*state*, by contrast, is internal and trusted, so the lighter `TypedDict` suffices. Right tool, right
layer — again.

`.model_dump()` turns the validated object back into a plain dict (Pydantic v2's method; v1 called
it `.dict()`) so the rest of the pipeline works with dicts. Every provider returns
`List[dict]`, all in this identical shape.


### Semantic Scholar & OpenAlex: rate-limits, and the fast/deep PDF ladder

These two are meatier. Two ideas dominate them:

**(1) Surfacing *why* a source was empty.** Semantic Scholar's free tier throttles hard (HTTP 429).
Rather than let a rate-limited source look like "no results," it retries with backoff and writes an
explicit status into a caller-provided `status_out` dict:


In [ ]:
# app/modules/search/providers/semantic_scholar.py  (rate-limit handling, trimmed)
response = None
for attempt in range(3):
    response = requests.get(url, headers=headers, params=params, timeout=15)
    if response.status_code != 429:
        break
    wait = 2 * (attempt + 1)                       # 2s, 4s, 6s backoff
    log.warning("Semantic Scholar rate-limited (429) — attempt %d/3, retrying in %ds", attempt+1, wait)
    time.sleep(wait)

if response.status_code == 429:
    if status_out is not None:
        status_out.update(state="rate_limited", http=429)     # <-- caller learns the real reason
    return []


That `status_out` dict is *mutated in place* — a deliberate **out-parameter** so the node can pull
the status back out after the call (`return {'semantic_scholar_status': status}`). It flows all the
way to the UI's status strip so the user sees "Semantic Scholar ⚠ rate-limited (HTTP 429)" instead
of a mysterious blank. Making failure **visible** is a maturity signal.

**(2) The HYBRID PDF strategy — the cleverest performance idea in the repo.** Getting a *real,
downloadable* PDF for a paper is expensive: the API's "open access URL" is often a landing page, not
a PDF. Verifying means downloading bytes and checking the file starts with the magic number `%PDF`,
then maybe scraping a `citation_pdf_url` meta tag, then maybe asking Unpaywall… per paper, that's
several network round-trips. Doing it for 25–40 candidates × 3 sources at *search* time would be
brutally slow. So NOVA splits it in two:


In [ ]:
# semantic_scholar.py — the FAST path (zero network) used during search
def resolve_pdf_url_fast(external_ids, fallback_url):
    """Cheap best-effort link, NO downloading/verifying. Deferred deep-verify to 'Chat it out'."""
    external_ids = external_ids or {}
    if external_ids.get("ArXiv"):
        return f"https://arxiv.org/pdf/{external_ids['ArXiv']}"        # deterministic repo URL
    if external_ids.get("ACL"):
        return f"https://aclanthology.org/{external_ids['ACL']}.pdf"
    if external_ids.get("PubMedCentral"):
        pmc = external_ids['PubMedCentral']
        clean = pmc if str(pmc).upper().startswith("PMC") else f"PMC{pmc}"
        return f"https://europepmc.org/articles/{clean}?pdf=render"
    return fallback_url or None                                        # else the API's own guess

# and the DEEP path (verify + scrape + Unpaywall), used later at chat time
def resolve_pdf_url(external_ids, fallback_url, unpaywall_email=None, log=None):
    # 1) ArXiv/ACL/PMC deterministic URLs, each VERIFIED by downloading 16 bytes & checking %PDF
    # 2) openAccessPdf.url verified directly, else scrape its citation_pdf_url meta tag
    # 3) DOI -> Unpaywall best_oa_location.url_for_pdf   (only if an email is configured)
    ...


So during **search** (`fast=True`, the default), every candidate gets a cheap guessed link with
**no network calls** and *all* papers are kept — search stays snappy. The expensive verification is
**deferred to "Chat it out" time**, when the user has picked *one* paper and we only need to nail
*that* PDF. OpenAlex mirrors the exact same `fast`/deep split (`resolve_openalex_pdf_fast` vs
`resolve_openalex_pdf`), plus a neat `decode_abstract` that rebuilds text from OpenAlex's
*inverted-index* abstract format. This "cheap now, expensive on demand" is a textbook
**latency-vs-completeness** trade-off — exactly the kind of thing to raise proactively in an
interview.

One more detail worth admiring: verification checks the **actual response bytes** for `%PDF`, not
the `Content-Type` header or a field named "pdf" — because servers lie about content types all the
time. Trust the bytes, not the label.

The `decode_abstract` helper is **🔌 runnable-standalone** — a fun little algorithm to be able to
explain:


In [ ]:
# app/modules/search/providers/openalex.py  (decode_abstract) — RUNNABLE STANDALONE
def decode_abstract(inverted_index):
    """OpenAlex ships abstracts as {word: [positions]}. Rebuild the linear text."""
    if not inverted_index:
        return None
    try:
        max_position = max(pos for positions in inverted_index.values() for pos in positions)
    except ValueError:
        return None                                  # keys existed but all position lists empty
    words = [""] * (max_position + 1)
    for word, positions in inverted_index.items():
        for position in positions:
            words[position] = word
    result = " ".join(words).strip()
    return result if result else None

demo = {"Reinforcement": [0], "learning": [1], "improves": [2], "fuel": [3], "efficiency": [4]}
print(decode_abstract(demo))          # -> "Reinforcement learning improves fuel efficiency"


## 8. Aggregate → rerank → cluster: turning 3 lists into grouped results

Three provider lists come back, likely with overlap (a hot paper shows up on all three). The next
three nodes clean and organize them.

### Aggregation = dedup by normalized title, merge without losing anything

Two sources may title the same paper slightly differently ("Eco-Follower:" vs "EcoFollower"). We
need them to collapse into one record. The trick: a **normalized title** used *only* as a matching
key, never stored. This is **🔌 runnable-standalone**:


In [ ]:
# app/modules/search/graph.py  (normalize_title) — RUNNABLE STANDALONE
import re, unicodedata

def normalize_title(title: str) -> str:
    """Dedup key only — never stored on the record."""
    if not title:
        return ""
    text = unicodedata.normalize("NFKD", title)                 # split accents from letters
    text = "".join(c for c in text if not unicodedata.combining(c))  # drop the accent marks
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)                         # strip punctuation
    text = re.sub(r"\s+", " ", text)                           # collapse whitespace
    return text.strip()

for t in ["Eco-Follower: An RL Model", "EcoFollower  an   RL model", "Éco-Follower: an RL Model"]:
    print(repr(normalize_title(t)))


All three variants collapse to the same key. The `unicodedata.normalize("NFKD", ...)` +
drop-combining-characters combo is the standard way to fold accented characters (`é → e`, curly
quotes → straight) so titles compare equal across sources that encode Unicode differently.

Once titles match, `merge_paper_entry` fuses two records **without dropping information**. The key
idea: fields that legitimately *differ per source* (URL, PDF link, citation count) are stored as
**dicts keyed by source**, so nothing overwrites anything:


In [ ]:
# app/modules/search/graph.py  (merge_paper_entry — the important half)
def merge_paper_entry(existing: dict, new: dict) -> dict:
    new_source = new.get("source")
    # title: keep the longer/more complete one
    merged_title = (new.get("title") if len(new.get("title") or "") > len(existing.get("title") or "")
                    else existing.get("title"))
    # authors: order-preserving union
    merged_authors = list(existing.get("authors") or [])
    for a in (new.get("authors") or []):
        if a not in merged_authors:
            merged_authors.append(a)
    # source-VARYING fields: dict keyed by source, nothing overwritten
    merged_url = dict(existing.get("url") or {}); merged_url[new_source] = new.get("url")
    merged_pdf_url = dict(existing.get("pdf_url") or {}); merged_pdf_url[new_source] = new.get("pdf_url")
    merged_citation = dict(existing.get("citation_count") or {}); merged_citation[new_source] = new.get("citation_count")
    merged_sources = list(existing.get("source") or [])
    if new_source not in merged_sources:
        merged_sources.append(new_source)
    return {"title": merged_title, "authors": merged_authors or None, "abstract": ...,
            "year": ..., "source": merged_sources, "citation_count": merged_citation,
            "url": merged_url, "pdf_url": merged_pdf_url}


So after aggregation a paper found on arXiv **and** OpenAlex has `source: ["arXiv", "OpenAlex"]`
and `url: {"arXiv": ..., "OpenAlex": ...}`. Crucially, the *first-seen* single-source records are
wrapped into the **same** dict-keyed-by-source shape (see `aggregator_node`), so **every** record
has a uniform structure whether it came from one source or three. That uniformity is what lets the
UI's helper `first_available(record["pdf_url"])` blindly pull "any PDF link we've got" without
special-casing. Consistency of shape is what keeps the downstream code simple.

### Reranking with SPECTER: embeddings, cosine similarity, top-15

Aggregation gives ~20–40 unique papers; that's too many, and API "relevance" is weak. The reranker
scores each paper's *abstract* against the *research intent* using **SPECTER**
(`allenai-specter`) — an embedding model trained specifically on academic title/abstract pairs, so
it "understands" paper-to-paper similarity far better than a generic sentence embedder.


In [ ]:
# app/modules/search/reranking.py  (the math, trimmed)
from sentence_transformers import SentenceTransformer
import numpy as np

_MODEL = SentenceTransformer("sentence-transformers/allenai-specter")   # loaded ONCE at import

def rerank_by_relevance(research_intent, papers, top_n=15, logger=None):
    # papers: {normalized_title: abstract}. Skip empty abstracts (can't embed them).
    valid_titles, valid_abstracts = [], []
    for t, ab in papers.items():
        if ab and ab.strip():
            valid_titles.append(t); valid_abstracts.append(ab)
    if not valid_abstracts:
        return {}

    # normalize_embeddings=True => unit vectors => dot product IS cosine similarity
    query_vec  = _MODEL.encode(research_intent, normalize_embeddings=True, convert_to_numpy=True)
    paper_vecs = _MODEL.encode(valid_abstracts, normalize_embeddings=True, convert_to_numpy=True,
                               batch_size=32)
    similarities = paper_vecs @ query_vec            # (N,) dot products, one shot
    ranked = np.argsort(-similarities)               # descending: negate then ascending argsort
    top = ranked[:top_n]
    return {valid_titles[i]: valid_abstracts[i] for i in top}   # dict preserves ranked order


Three things to be able to say about this:

1. **Why `normalize_embeddings=True` matters.** Cosine similarity = dot product **only when both
   vectors are unit length**. Normalizing at encode time means a plain matrix multiply
   `paper_vecs @ query_vec` *is* the cosine similarity — no separate cosine call, and it's fully
   vectorized (all N papers scored in one NumPy op instead of a Python loop).
2. **`np.argsort(-similarities)`.** `argsort` sorts *ascending* and returns *indices*; negating the
   scores flips it to descending. So `ranked[0]` is the index of the most-similar paper. Slicing
   `[:top_n]` keeps the best 15.
3. **Dict insertion order = ranking.** Since Python 3.7 dicts keep insertion order, building the
   result dict in ranked order means *iterating it later yields papers best-first*, for free.

`_MODEL` is loaded at **module import**, not per call — and that's the very side-effect that
`load_agents()`/`jobs.py` rely on to warm SPECTER once at boot. Empty-abstract papers are excluded
from ranking (you can't embed nothing) but *logged*, so nothing vanishes silently.

**Two rerankers, don't confuse them:** SPECTER here is a **bi-encoder** (embed intent and abstracts
separately, compare vectors — fast, scales to many papers). The chatbot's `bge-reranker-base`
(Section 10) is a **cross-encoder** (feed query+passage *together*, get one relevance score — slower
but sharper, used on a handful of chunks). Bi-encoder to *filter many*, cross-encoder to *sharpen
few*: a classic two-stage retrieval pattern, and NOVA uses both, at different layers.


### Clustering with structured output: making the LLM return *typed* data

The final node groups the 15 papers by **approach** ("RL-based control policies," "physics-based
models," …) — not by topic (they already share a topic; that's why they were retrieved). The
challenge: an LLM naturally returns prose, but we need a *parseable* structure. Enter
**`with_structured_output`**:


In [ ]:
# app/modules/search/graph.py  (clustering — the structured-output core)
from pydantic import BaseModel, Field
from typing import List

class ClusterItem(BaseModel):
    cluster_id: int
    label: str = Field(description="Short, specific approach name")
    rationale: str = Field(description="2-3 sentence explanation of what unifies this cluster")
    paper_ids: List[int]

class ClusteringOutput(BaseModel):
    clusters: List[ClusterItem]

def clustering_node(state):
    logger = get_node_logger(state["run_id"], "search.cluster")
    clustering_input, id_to_title = prepare_papers_for_clustering(state['reranked_papers'])
    prompt = f"""Organize these papers into methodologically distinct APPROACH CLUSTERS...
    ... every paper ID appears in exactly one cluster; use an 'Other / Mixed' catch-all if needed ...
RESEARCH INTENT:
{state["ResearchIntent"]}
PAPERS:
{clustering_input}"""

    structured_llm = llm_2.with_structured_output(ClusteringOutput)   # <-- force typed output
    try:
        clustering_result = structured_llm.invoke(prompt)             # returns a ClusteringOutput
    except Exception as e:
        logger.exception("Structured LLM invocation failed during clustering")
        return {"clustered_papers": []}                               # degrade gracefully, don't crash

    clustered = build_clustered_output(clustering_result, id_to_title, state["aggregator"])
    return {"clustered_papers": clustered}


**`with_structured_output(ClusteringOutput)`** binds a Pydantic schema to the LLM so the call
returns a **validated `ClusteringOutput` object**, not a string. Under the hood LangChain does this
via the model's tool/function-calling or JSON-mode support and validates the result against the
schema. This is *dramatically* more robust than the old way — "return JSON" in the prompt, then
`json.loads()` the reply and pray. You get typed attributes (`clustering_result.clusters[0].label`)
and validation for free.

Notice the two defensive layers around the LLM's fallibility:

- **The IDs are indirection.** The prompt sends papers as small integer IDs (0, 1, 2…), and
  `id_to_title` maps them back to real records. Why not send full titles/records? Cheaper prompt,
  and it lets us *validate* what comes back.
- **Hallucinated-ID guard.** In `build_clustered_output`, if the LLM invents a `paper_id` that was
  never sent, `id_to_title.get(pid)` returns `None` and that entry is **skipped**, not crashed on.
  Same for a missing aggregator record. The code assumes the LLM *will* occasionally misbehave and
  refuses to trust it blindly.
- **Whole-node fallback.** If the structured call throws entirely, the node returns
  `{"clustered_papers": []}` — the search still "succeeds" with zero clusters, and the UI shows a
  friendly "couldn't pull solid matches" screen instead of a stack trace.

Also note `prepare_papers_for_clustering` **excludes papers with no abstract** from the clustering
input (a title alone is too weak a grouping signal) — a quiet quality decision.


### Streaming the search graph so the UI ticks off steps live

Back in the UI, `render_searching` doesn't `.invoke()` the search graph (which would hang on one
spinner for ~30s). It **`.stream()`s** it with `stream_mode="updates"`, so each node's completion
arrives as an event and the UI checks that step off a live list:


In [ ]:
# utils/view_searching.py  (streaming half)
completed, counts, final_state = set(), {}, {}
source_field = {"arxiv": "arXiv_paper", "semantic_scholar": "Semantic_Scholar_paper",
                "open_alex": "Open_Alex_paper"}
for update in search_graph.stream(
        {"ResearchIntent": human_verified_intent, "run_id": run_id}, stream_mode="updates"):
    for node_name, delta in update.items():        # updates mode: {node_name: partial_state}
        completed.add(node_name)
        if isinstance(delta, dict):
            final_state.update(delta)              # accumulate every node's output
            if node_name in source_field:
                counts[node_name] = len(delta.get(source_field[node_name]) or [])
    board.markdown(search_steps_html(completed, counts), unsafe_allow_html=True)  # repaint checklist

clusters = final_state.get("clustered_papers") or []
# flatten every paper into papers_by_key[norm_title] = record  (so "Chat it out" can find any paper)
papers_by_key = {}
for cluster in clusters:
    for norm_title, record in (cluster.get("papers") or {}).items():
        papers_by_key[norm_title] = record


**`stream_mode="updates"`** yields, after each node finishes, a dict `{node_name: the_partial_state_that_node_returned}`.
(Contrast **`stream_mode="values"`**, which yields the *entire accumulated state* each time — more
data, and you'd have to diff it yourself to know what changed.) `"updates"` is exactly right for a
progress checklist: each event tells you precisely which step just completed and what it produced,
so `counts["arxiv"] = len(delta["arXiv_paper"])` gives the live per-source paper count. NOVA
accumulates the deltas into `final_state` itself since it wants the *final* clusters at the end.

`search_steps_html` (in `utils/search_progress.py`) is a pure renderer: given the set of completed
node names and the counts, it returns the checklist HTML with ✓/•/blank states. Keeping the
"compute the view string" logic in its own pure function — separate from the Streamlit calls — makes
it trivially testable, a small but nice separation-of-concerns choice.

The **flatten** step at the end matters for the next feature: results are nested
(`clusters → papers → record`), but "Chat it out" needs to look up *any* paper by its key regardless
of which cluster it's in. So everything is copied flat into `papers_by_key[norm_title] = record`.


## 9. The other front door: FastAPI (present in `app/`, not used by NOVA)

Here's a subtlety that impresses interviewers *if you can articulate it*. The `app/` package was
lifted from a standalone service and still carries a **complete FastAPI HTTP API** —
`app/main.py` plus `router.py` in each module. **NOVA never runs it** (there's no uvicorn in
`run.sh`; the Streamlit UI imports the *compiled graphs* directly). But it's worth understanding,
because it shows the graphs are **UI-agnostic**: the same INTENT/SEARCH graphs can be driven either
by Streamlit *or* by HTTP, and the "drive" code is thin either way.


In [ ]:
# app/modules/intent/router.py  (the HTTP mirror of what the Streamlit views do)
@router.post("/start", response_model=IntentStartResponse)
def start_intent(payload: IntentStartRequest):
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    result = intent_graph.invoke({"user_query": payload.user_query, "run_id": thread_id}, config=config)
    interrupt_payload = result["__interrupt__"][0].value          # same interrupt pattern as the UI
    return IntentStartResponse(thread_id=thread_id, **interrupt_payload)

@router.post("/resume", response_model=IntentResumeResponse)
def resume_intent(payload: IntentResumeRequest, background_tasks: BackgroundTasks):
    result = intent_graph.invoke(Command(resume=payload.edited_intent), config=...)  # same resume pattern
    human_verified_intent = result["human_verified_intent"]
    job_id = str(uuid.uuid4())
    create_job(job_id, payload.thread_id)
    background_tasks.add_task(run_search_job, job_id, human_verified_intent, payload.thread_id)
    return IntentResumeResponse(status="running", job_id=job_id, ...)


Map it to what you already know: `/intent/start` is the HTTP twin of `render_refining`
(invoke → hit the interrupt → return the draft), and `/intent/resume` is the twin of
`render_searching` (resume with edited text → launch the search). The difference is *how the slow
search runs*:

- **In FastAPI:** it's a **`BackgroundTask`**. The `/resume` call returns *immediately* with a
  `job_id` and `status: "running"`; the client then **polls** `GET /search/status/{job_id}` until
  it flips to `done`. Progress lives in a plain in-memory `JOBS` dict (`app/modules/search/jobs.py`).
- **In Streamlit:** there's no polling — the single process just **streams** the graph inline and
  repaints (Section 8). Simpler, because one browser session maps to one running script.

`app/modules/search/jobs.py` carries the same import-time-side-effect trick as `load_agents`: it
does `from app.modules.search.graph import graph as search_graph` at module top *specifically so
that importing jobs.py at FastAPI startup warms the SPECTER model once*, not on the first user's
request. Same reasoning as NOVA's `boot()`, different host. The FastAPI `main.py` also sets up root
logging and permissive dev CORS (`allow_origins=["*"]`, with a comment telling you to lock it down
before production — an honest TODO).

The Pydantic **request/response schemas** (`IntentStartRequest`, `SearchStatusResponse`, …) are the
*third* place Pydantic shows up: here it powers FastAPI's automatic request validation and OpenAPI
docs. Same library, three jobs — graph-output typing, provider-data validation, HTTP contract.


## 10. "Chat it out": a retrieval-augmented Q&A bot over one PDF

Click **💬 Chat it out** on a paper and NOVA switches to its third program: a single-PDF **RAG**
(Retrieval-Augmented Generation) chatbot. RAG in one sentence: *don't ask the LLM what it knows —
fetch the relevant passages from the source and make it answer strictly from those.* That's what
keeps answers grounded and citable.

Two files: `vectorizeer.py` (PDF → searchable vector store) and `Qa.py` (retrieve → prompt → stream).

### Step 1: PDF → semantic chunks → Chroma vector store

You can't embed a whole 12-page PDF as one vector; you **chunk** it, embed each chunk, and store the
vectors so you can later fetch the few chunks nearest a question. *How* you chunk hugely affects
answer quality. `vectorizeer.py` is smart about it — three strategies, in order:


In [ ]:
# chatbot_core/vectorizeer.py  (chunking pipeline, trimmed)
import re, os, hashlib
import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Matches "1 Introduction", "3.2 Inference with V-RAG", "6 Conclusion"
HEADER_PATTERN = re.compile(r'\n(\d{1,2}(?:\.\d{1,2})?\s+[A-Z][A-Za-z][^\n]{2,60})(?=\n)')

def get_pdf_hash(pdf_path):
    with open(pdf_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()[:12]   # content hash -> stable per-PDF cache dir

def build_vectorstore(pdf_path, persist_root="./vectorstores", force_rebuild=False):
    persist_dir = os.path.join(persist_root, get_pdf_hash(pdf_path))
    embeddings = get_embeddings()
    if os.path.isdir(persist_dir) and os.listdir(persist_dir) and not force_rebuild:
        return Chroma(persist_directory=persist_dir, embedding_function=embeddings)  # cache hit

    sections = extract_sections(pdf_path)               # 1) split by detected section headers
    splitter = RecursiveCharacterTextSplitter(          # 3) fallback splitter for oversized prose
        chunk_size=700, chunk_overlap=100, separators=["\n\n", "\n", ". ", " "])
    docs = []
    for sec in sections:
        for is_table, piece_text in _split_protecting_tables(sec["text"], splitter):  # 2) keep tables intact
            docs.append(Document(page_content=piece_text,
                metadata={"section": sec["title"], "page": sec["page"],
                          "type": "table" if is_table else "text",
                          "source": os.path.basename(pdf_path)}))
    if not docs:
        raise ValueError("No extractable text — this PDF may be scanned (image-only) and need OCR.")
    return Chroma.from_documents(docs, embeddings, persist_directory=persist_dir)


**The chunking philosophy — why not just cut every 700 characters?** Because an arbitrary cut
splits a sentence mid-thought and shreds tables. NOVA instead:

1. **Splits by section headers** (`HEADER_PATTERN` finds "1 Introduction", "3.2 ...") so each chunk
   is a real semantic unit — and each chunk remembers its section + page number for **citations**.
2. **Protects tables**: `_split_protecting_tables` detects lines that are mostly numbers and keeps
   consecutive table-ish lines together as one chunk, so a results table isn't sliced into
   meaningless fragments.
3. **Falls back to `RecursiveCharacterTextSplitter`** only for oversized prose blocks. That splitter
   tries separators in order — paragraph (`\n\n`) → line (`\n`) → sentence (`. `) → word (` `) —
   cutting at the *coarsest* boundary that fits the size limit. `chunk_overlap=100` repeats ~100
   chars between adjacent chunks so a fact straddling a boundary isn't lost.
4. And if **no headers** are found at all (weird formatting), it falls back again to page-wise chunks
   with overlap — *"nothing is silently lost"* is the recurring principle.

**Content-hash caching.** `get_pdf_hash` is the MD5 of the file's *bytes*, so the persist directory
is stable per PDF. Re-open the same paper and `build_vectorstore` **loads** the existing Chroma
store instantly instead of re-embedding. (Different from the URL-hash cache in `papers.py` — one
keys on *file content*, the other on *download URL*. Two caches, two appropriate keys.)

**Embeddings:** `BAAI/bge-base-en-v1.5`, a strong ~440 MB local model — *no API key, no rate limits,
no per-call cost*, runs on CPU. **Chroma** is the vector DB: `Chroma.from_documents(...)` embeds
every chunk and persists the vectors + metadata to disk. The image-only-PDF guard raises a clear,
actionable error ("needs OCR first") instead of building an empty, useless store.


### Step 2: retrieve → rerank → grounded, streaming answer

`Qa.py` turns that vector store into an answering chain. The retrieval is **two-stage**, and the
prompt is a strict "answer only from these excerpts" contract:


In [ ]:
# chatbot_core/Qa.py  (retriever + chain, trimmed)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

def get_llm():
    api_key = os.environ.get("SECOND_GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found. Add it to a .env file ...")
    return ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

def get_retriever(vectorstore, k=15, top_n=5):
    base = vectorstore.as_retriever(search_kwargs={"k": k})          # 1) bring 15 by vector similarity
    reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
    compressor = CrossEncoderReranker(model=reranker, top_n=top_n)   # 2) cross-encode -> keep best 5
    return ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Section: {d.metadata.get('section','?')} | Page: {d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs)

def build_chain(llm):
    prompt = ChatPromptTemplate.from_messages([
        ("system", QA_SYSTEM_PROMPT),          # "Ground every answer strictly in the excerpts... cite pages"
        MessagesPlaceholder("chat_history"),   # slot for prior turns
        ("human", "{question}"),
    ])
    return prompt | llm                        # LCEL: pipe the prompt into the model


**Two-stage retrieval, the accuracy lever.** `get_retriever` fetches `k=15` candidate chunks by
cheap vector similarity, then a **cross-encoder** (`bge-reranker-base`) re-scores each *(question,
chunk)* pair *together* and keeps only the `top_n=5`. As noted in Section 8, a cross-encoder is
slower but far more accurate than bi-encoder cosine — and here it only runs on 15 chunks, so the
cost is fine. `ContextualCompressionRetriever` is the LangChain wrapper that composes "base retriever
→ compressor(reranker)" into one `retriever.invoke(question)` call. This retrieve-many-then-rerank-few
is exactly the two-stage pattern, now at the *chunk* level.

**`prompt | llm` — that pipe is LCEL** (LangChain Expression Language). The `|` composes Runnables
left-to-right: the prompt template formats the messages, the result flows into the LLM. It's the
modern replacement for the old `LLMChain` class — more composable, streamable, and inspectable.
`ChatPromptTemplate` with a **`MessagesPlaceholder("chat_history")`** reserves a slot that gets
filled with prior turns at call time, so the bot has conversation memory. `temperature=0` makes
answers deterministic/factual — right for grounded Q&A where you don't want creative drift.

> **⚠️ Bug watch (real, worth calling out):** `get_llm()` reads **`SECOND_GROQ_API_KEY`** for its
> presence check, but the error message says **`GROQ_API_KEY`**, *and* it constructs
> `ChatGroq(model=..., temperature=0)` with **no `api_key=` argument** — so the client silently
> falls back to reading the **`GROQ_API_KEY`** env var itself. Net effect: the guard validates one
> key (`SECOND_GROQ_API_KEY`) while the LLM actually authenticates with a *different* one
> (`GROQ_API_KEY`). If a user set `SECOND_GROQ_API_KEY` but not `GROQ_API_KEY`, the guard passes and
> then the real call 401s; if they set only `GROQ_API_KEY`, the guard *raises* even though the call
> would have worked. In *this* repo both keys exist in `.env`, so it works by luck — but the guard
> and the credential being decoupled is a genuine latent bug. (The intent, comparing to the graphs,
> was almost certainly `ChatGroq(..., api_key=api_key)`.)


### Streaming answers with citations, in the Streamlit chat view

`utils/view_chat.py` wires the retriever + chain into a chat UI: replay history, take a question,
retrieve, stream tokens, then print the page citations.


In [ ]:
# utils/view_chat.py  (the answer path, trimmed)
from langchain_core.messages import HumanMessage, AIMessage
from Qa import format_docs

prompt = st.chat_input("Ask about this paper…")
if prompt:
    session["messages"].append({"role": "user", "content": prompt})
    history = [HumanMessage(content=m["content"]) if m["role"] == "user" else AIMessage(content=m["content"])
               for m in session["messages"][:-1]]      # all prior turns, excluding the new question
    with st.chat_message("assistant", avatar=SONIC_DATA_URI):
        docs = session["retriever"].invoke(prompt)      # 2-stage retrieve for THIS question
        context = format_docs(docs)
        def token_stream():
            for chunk in session["chain"].stream(
                    {"question": prompt, "chat_history": history, "context": context}):
                if chunk.content:
                    yield chunk.content                 # yield tokens as they arrive
        answer = st.write_stream(token_stream())        # live-render the stream
        pages = sorted({f"p.{d.metadata.get('page')}" for d in docs})
        if pages:
            st.caption("sources: " + ", ".join(pages))  # citations from chunk metadata
    session["messages"].append({"role": "assistant", "content": answer})


`st.write_stream` consumes a generator and renders tokens live — the ChatGPT-style typing effect.
The `context` (formatted excerpts) is injected into the system prompt's `{context}` slot, so the LLM
answers *only* from retrieved text. Citations come straight from each chunk's `page` metadata — the
same metadata attached during chunking, closing the loop from PDF page → chunk → answer citation.
`chat_history` is rebuilt from `session["messages"][:-1]` (everything *before* the just-added
question), converted to LangChain `HumanMessage`/`AIMessage` objects for the `MessagesPlaceholder`.

> **⚠️ Bug watch (small, cosmetic — but it's in two places):** `sorted({f"p.{page}" ...})` sorts
> the page tags as **strings**, so `"p.10"` sorts *before* `"p.2"` (lexicographic, not numeric).
> Cited pages can appear out of order (p.1, p.10, p.2, p.3…). Same issue in `Qa.py`'s CLI `main()`.
> Harmless to correctness, but a real ordering bug — the fix is a numeric key,
> e.g. `sorted(pages, key=lambda s: int(s[2:]))`.


### Why the slow prep runs in a background thread

Downloading + vectorizing a PDF takes several seconds. If that ran on Streamlit's main thread, the
UI would freeze. `utils/chat_engine.py` runs it in a **daemon thread** while the main thread animates
a progress bar + rotating SONIC quote:


In [ ]:
# utils/chat_engine.py  (the threading pattern, trimmed)
import threading, time, random

def _prepare_chat_blocking(record, holder):
    """Background worker. Touches NO st.* (thread-safe). Results go in `holder`."""
    try:
        pdf_path = resolve_and_download_pdf(record)                 # the DEEP PDF resolve (Section 7)
        if not pdf_path:
            holder["error"] = "Couldn't find a readable open-access PDF for this paper…"
            return
        from vectorizeer import build_vectorstore
        from Qa import get_llm, get_retriever, build_chain
        vs = build_vectorstore(pdf_path)
        holder["session"] = {"retriever": get_retriever(vs), "chain": build_chain(get_llm())}
    except Exception as e:
        holder["error"] = f"Couldn't prepare this paper for chat: {type(e).__name__}: {e}"

def ensure_chat_ready(norm_title):
    if norm_title in st.session_state.chats:
        return st.session_state.chats[norm_title], None            # already built -> instant
    record = st.session_state.papers_by_key.get(norm_title, {})
    holder = {}
    worker = threading.Thread(target=_prepare_chat_blocking, args=(record, holder), daemon=True)
    worker.start()
    pct, start = 6, time.time()
    while worker.is_alive():                                        # main thread animates while worker runs
        pct = min(pct + 2, 92)
        # ...update progress bar + rotate a SONIC quote every ~4s...
        time.sleep(0.35)
    worker.join()
    if holder.get("error"):
        return None, holder["error"]
    session = holder["session"]; session.update({"messages": [], "title": ...})
    st.session_state.chats[norm_title] = session                   # cache in session for instant reopen
    return session, None


The critical rule, stated in the code's own docstring: **the worker touches no `st.*`.**
Streamlit's `session_state` and widget calls are bound to the main thread's `ScriptRunContext`;
calling them from a spawned thread is unsupported and flaky. So the pattern is the classic
**mutable `holder` handoff**: the thread writes its result into a plain dict, the main thread reads
it back after `join()`. All Streamlit rendering (progress bar, quotes) stays on the main thread.

This is also where the **deep** PDF resolution finally happens — `resolve_and_download_pdf` runs the
expensive verify/scrape ladder we *deferred* back in search (Section 7). By now the user picked one
paper, so paying that cost for a single PDF is exactly the right moment. The built
`{retriever, chain, messages, title}` session is cached in `st.session_state.chats[norm_title]`, so
reopening the same paper's chat is instant (`ensure_chat_ready`'s first line).

`build_vectorstore(pdf_path)` here relies on `os.chdir(BASE)` from Section 1: its default
`persist_root="./vectorstores"` resolves under `NOVA/` *only because* `paths.py` changed the working
directory at import time. That one line from the very beginning is what makes this line, ten sections
later, write to the right place. The whole app really is one connected system.


## 11. Two supporting systems: per-run logging & the SONIC persona

### Per-run logging: one file tells one query's whole story

Every query gets a `run_id` (the UUID minted in `render_welcome`). It flows through *both* graphs as
`run_id` and *every* node, provider, and reranker logs through `get_node_logger(run_id, node_name)`.
The result: `logs/{run_id}.log` contains the complete, in-order story of exactly one research query.


In [ ]:
# app/core/logging_config.py  (the important guard)
def _get_run_logger(run_id):
    if run_id in _run_loggers:
        return _run_loggers[run_id]
    logger = logging.getLogger(f"run.{run_id}")
    logger.setLevel(logging.INFO)
    logger.propagate = False                       # (a) don't double-print via the root logger
    if not logger.handlers:                        # (b) don't stack duplicate handlers on re-setup
        fh = logging.FileHandler(LOG_DIR / f"{_safe_filename(run_id)}.log", encoding="utf-8")
        fh.setFormatter(_FORMATTER); logger.addHandler(fh)
        logger.addHandler(logging.StreamHandler())
    _run_loggers[run_id] = logger
    return logger

def get_node_logger(run_id, node_name):
    return _get_run_logger(run_id).getChild(node_name)   # e.g. run.<id>.search.arxiv


Two lines encode hard-won logging lessons — the sort of thing that shows real Python depth:

- **`propagate = False`**: Python loggers bubble records up to ancestor loggers by default. The
  FastAPI `main.py` configures a root handler; without `propagate=False`, every line would print
  *twice* (once here, once via root). Turning off propagation keeps per-run logs isolated.
- **`if not logger.handlers:`**: `logging.getLogger(name)` returns the **same** logger object for a
  given name, process-wide. In a host that re-executes the module (Streamlit reruns, or a reused
  `run_id`), naively re-adding handlers would stack them, and every line would be written N times.
  Guarding on "does it already have handlers?" makes setup **idempotent**. The `_run_loggers` dict
  is a second layer of the same defense.

`getChild(node_name)` gives a hierarchical name like `run.<id>.search.arxiv`, so each log line names
the exact node — searchable, and it inherits the parent's file+console handlers so you don't
re-wire them per node.

### SONIC: an inline SVG persona (no external assets)

`utils/sonic.py` draws SONIC as an **inline SVG**, base64-encoded into a `data:` URI. Why not a PNG
file? Streamlit's strict Content-Security-Policy blocks external asset requests; a `data:` URI is
*embedded*, so it always renders — same reason the artifact system inlines everything. `sonic_says()`
is just a styled speech-bubble helper the views call to keep the persona talking. It's pure
presentation, but it's the "S" in the product's story, so worth knowing it's a self-contained
data-URI, not a file on disk.


## 12. Synthesis — the whole system as one mental model

Trace a single query through everything you now know, and notice how each idea handed off to the next:

1. **`run.sh` → `nova_app.py`**, and the very first import (`utils.paths`) fixes `sys.path` + `chdir`
   + `.env` so the two copied sub-projects resolve unchanged. *(Import-time side-effects, module
   cache — §1.)*
2. Streamlit **reruns the whole script** on every interaction, so heavy models load once via
   **`@st.cache_resource`** behind `boot()`, and a **stage FSM** in `session_state` decides which
   view renders. *(Rerun model, caching, run-once guards — §2–4.)*
3. The **INTENT graph** frames the idea with three parallel single-purpose LLM nodes (**fan-out/
   fan-in**), grounds it via a **Tavily**-augmented node, and **pauses at `interrupt()`** — resumed
   across three screens by one `thread_id` + an in-memory **checkpointer**. *(LangGraph core, HITL —
   §5.)*
4. A tiny **regex round-trip** keeps the graph's three-section text and the UI's three form fields
   in sync, honoring the literal format contract both graphs' prompts depend on. *(§6.)*
5. The **SEARCH graph** fans out to three APIs — each queried in its *own dialect* by an LLM
   translator — writing to **disjoint state keys** to dodge concurrent-write conflicts; validates
   every record through the **`ResearchPaper`** Pydantic model; and defers expensive PDF verification
   via the **fast/deep** split. *(Parallel state, Pydantic-at-the-edge, latency trade-off — §7.)*
6. Results are **deduped** by normalized title, **reranked** by a SPECTER **bi-encoder** (cosine via
   normalized dot product, top-15), and **clustered** by an LLM returning validated
   **`with_structured_output`** — with hallucinated-ID and whole-node fallbacks. The UI **streams**
   the graph (`stream_mode="updates"`) to tick steps off live. *(Embeddings, structured output,
   streaming — §8.)*
7. The same graphs are independently drivable over **FastAPI** (background jobs + polling) — proof
   the agents are UI-agnostic. *(§9.)*
8. "Chat it out" builds a **RAG** bot: section-aware **chunking** → **BGE embeddings** → **Chroma**,
   then a two-stage **bi-encoder → cross-encoder** retrieve, an **LCEL `prompt | llm`** chain, and
   **streamed, page-cited** answers — with the slow prep pushed to a **background thread** that never
   touches `st.*`. *(RAG, two-stage retrieval, LCEL, threading — §10.)*
9. Throughout, **per-run logging** (idempotent handlers, `propagate=False`) records one file per
   query, and **SONIC** narrates via inline-SVG data URIs. *(§11.)*

**The two recurring patterns that define NOVA's engineering personality**, worth naming explicitly
in an interview:
- **"Load the expensive thing once."** Module-scope singletons, `@st.cache_resource`, import-time
  model warming, content-hash + URL-hash caches, cached chat sessions — the same instinct at six
  different layers.
- **"Never trust the boundary; degrade, don't crash."** Pydantic-validate every provider record,
  regex-fallback on malformed intent text, skip hallucinated cluster IDs, return `[]` on a failed
  node, surface rate-limits as visible status, guard against duplicate log handlers. The system
  assumes APIs and LLMs *will* misbehave and stays standing when they do.

**And the parts that are genuinely fragile** (all flagged above): the `SECOND_GROQ_API_KEY` /
`GROQ_API_KEY` guard-vs-credential mismatch in `Qa.get_llm` (§10), the unused `api_key` + duplicate
`llm_2` in the search graph (§7), the lexicographic page-sort (§10), the `+`/`replace` double-encode
in the arXiv query (§7), the unused return of `warm_chatbot_models` (§4), and the `agent/` vs `app/`
doc drift (§1). None are fatal; all are the kind of small thing a strong reviewer catches.


## 13. Hands-on exercises (reuse exactly what we covered)

Work these in NOVA's virtualenv. They're ordered easy → hard and each reinforces a specific concept.

**A. Prove the module-cache claim (§1).** Add a `print("paths loaded!")` at the bottom of
`utils/paths.py`, then in a fresh Python REPL run `import utils.paths` three times. It prints
**once**. Explain why in terms of `sys.modules`.

**B. Fix the page-sort bug (§10).** In `utils/view_chat.py`, replace
`sorted({f"p.{d.metadata.get('page')}" for d in docs})` with a version that sorts **numerically**
(hint: sort the raw page numbers first, then format). Verify p.2 now precedes p.10. Then find and fix
the twin bug in `Qa.py`'s `main()`.

**C. Fix the API-key bug (§10).** Change `Qa.get_llm()` so the key it *checks* is the key it *uses*
— pass `api_key=api_key` into `ChatGroq(...)` and make the variable name, the error message, and the
env var all agree. Decide deliberately whether the canonical key should be `GROQ_API_KEY` or
`SECOND_GROQ_API_KEY`, and justify it.

**D. Extend the state contract (§6).** Add a fourth field — *"Success Criteria"* — end to end: a new
form box in `view_review.py`, a new capture group in `intent_text._SECTION_PATTERN`, and a matching
section in `join_intent_sections`. Confirm the round-trip (`split` then `join`) is still lossless.
What breaks if you update `split` but forget `join`?

**E. Add a fourth source (§7).** Sketch a `providers/core.py` (CORE API, or a stub) exposing
`search_core(query, logger, status_out) -> List[dict]` returning `ResearchPaper.model_dump()` dicts.
Wire a `core_papers` node into the search graph with its **own** `core_status` state key. Which
existing functions need *zero* changes because of the uniform record shape? (Aggregation, rerank,
clustering, the whole UI — that's the payoff of the shared model.)

**F. Swap the streaming mode (§8).** In `view_searching.py`, change `stream_mode="updates"` to
`"values"` and print each yielded object's keys. Explain what changed and why `"updates"` is the
better fit for a progress checklist. What would you have to rewrite to keep the live counts working?

**G. Tune retrieval quality (§10).** In `Qa.get_retriever`, try `k=30, top_n=8`. Reason about the
accuracy-vs-latency and context-length trade-offs. Where does the cross-encoder cost grow, and where
doesn't it?

**H. Make the checkpointer durable (§5).** Swap `MemorySaver()` for a `SqliteSaver` in
`intent/graph.py`. What new capability does that unlock (hint: what survives a process restart mid-
review?), and why is it overkill for NOVA's single-user scope?


## 14. Every concept covered (your revision checklist)

**Streamlit & app architecture**
- Top-to-bottom **rerun-on-every-interaction** model; why it forces caching + guards
- **`st.session_state`** as cross-rerun memory; `setdefault` vs `=`; `type(v)()` to avoid shared mutables
- **Stage finite-state machine** + router; `st.rerun()` to advance; overlay ordering (chat over results)
- **`@st.cache_resource`** (singletons/models) vs **`@st.cache_data`** (serializable, copied); `show_spinner`
- Two-layer "load once": session `_booted` flag + process-wide resource cache
- **Run-once guards** keyed by `run_id`; `st.spinner`, `st.write_stream`, `st.chat_input`, `st.empty`, `st.progress`
- `unsafe_allow_html`, inline-SVG **`data:` URIs** under strict CSP

**Python / packaging mechanics**
- **Import-time side-effects**; `sys.path.insert`, `os.chdir`, `load_dotenv`; the **`sys.modules` cache**
- `# noqa: F401`; module-scope singletons; **mutable-default** aliasing footgun
- **Out-parameter** (`status_out`) mutation; content-hash vs URL-hash caching (MD5)
- **Daemon threads** + `holder` handoff; thread-safety (no `st.*` off the main thread)
- Idempotent **logging** setup: `propagate=False`, `if not logger.handlers`, `getChild`

**LangGraph**
- **`StateGraph`**, `TypedDict` state, nodes as `state -> partial_state`, `add_node`/`add_edge`, `START`/`END`
- **Fan-out / fan-in** parallelism; declarative barrier sync; **disjoint keys** to avoid concurrent-write conflicts (and when you'd need a **reducer**)
- **`interrupt()`** human-in-the-loop; **checkpointer** (`MemorySaver` and its limits); **`thread_id`**; **`Command(resume=...)`**; the **`__interrupt__`** payload
- **`.invoke()`** vs **`.stream()`**; `stream_mode="updates"` vs `"values"`
- **`with_structured_output(PydanticModel)`** for typed LLM output; hallucination guards + graceful `[]` fallback

**LLM / retrieval / RAG**
- **Prompt decomposition** (one concern per node); per-backend **query translation**; `temperature=0`
- **Augmented LLM** (Tavily tool node): LLM → tool → LLM
- **RAG**; section-aware + table-protecting **chunking**; `RecursiveCharacterTextSplitter`; `chunk_overlap`
- **Embeddings** (`BAAI/bge-base`, SPECTER); **Chroma** persist/load; **cosine = normalized dot product**; `np.argsort`
- **Bi-encoder vs cross-encoder**; two-stage retrieve-many-then-rerank-few; `ContextualCompressionRetriever` + `CrossEncoderReranker`
- **LCEL** `prompt | llm`; `ChatPromptTemplate` + `MessagesPlaceholder`; token **streaming**; **page citations** from chunk metadata

**Data modeling & robustness**
- **Pydantic** in three roles: graph structured-output, provider validation, FastAPI request/response; `Literal`, `Field`, `.model_dump()`
- **Uniform record shape** (dict-keyed-by-source) enabling simple downstream code
- Title **normalization** (Unicode NFKD) for dedup; loss-preserving **merge**
- **Fast/deep** PDF resolution (latency vs completeness); **`%PDF`** magic-byte verification; landing-page scraping; Unpaywall fallback

**Serving & ops**
- **FastAPI** routers, **`BackgroundTasks`**, in-memory job dict + **polling**; UI-agnostic graphs; dev **CORS**
- **XML/Atom** parsing (arXiv); **rate-limit** retry/backoff + visible status; OpenAlex **inverted-index** abstract decode

**Bugs & smells you can now spot** — guard-vs-credential key mismatch (`Qa.get_llm`), unused `api_key`
+ duplicate `llm_2`, lexicographic page-sort (×2), `+`/`replace` double-encode in the arXiv query,
unused `warm_chatbot_models` return, `agent/` vs `app/` doc drift.

---
*You now hold NOVA as one connected mental model — from the first `import` that bends `sys.path`, to
the last streamed token with its page citation. Walk an interviewer down §12 and you've told the
whole story.* ✦
